# PanCAKE Reference Exposure Optimization

This is a quick notebook to generate exposure times for a given reference star based on known exposure settings for a primary star using PanCAKE. Simply provide the target, reference, and your exposures settings.

The notebook will return a list of "optimal exposures" which are designed to reach a similar per-integration SNR. You can adjust the minimum number of integrations in the calls below if needed. Fewer integrations can lead to longer ramps, and less noise, but give less versatility in correct cosmic rays. Ideally 3-4 integrations should be enough to correct for most cosmic rays using temporal cleaning methods. The default minimum number of integrations is 4.

NOTE: Simbad was down when I uploaded this, I'm 99% sure it will work when it's restored.

In [5]:
import pancake

In [2]:
observations = {
    'Beta Pic': {
        'reference': 'Alpha Pic',
        'exposures': [
            ('F410M', 'SHALLOW4', 9, 4),
            ('F430M', 'MEDIUM8', 8, 4),
            ('F460M', 'MEDIUM8', 8, 5),
        ],
    },
    'HIP 65426': {
        'reference': 'HIP 68245',
        'exposures': [
            ('F430M', 'MEDIUM8', 10, 3),
            ('F460M', 'MEDIUM8', 10, 5),
        ],
    },
}

In [3]:
seqs = []
for target_name, obs in observations.items():
    target = pancake.scene.Scene('Target')
    target.add_source(target_name, kind='simbad')

    reference = pancake.scene.Scene('Reference')
    reference.add_source(obs['reference'], kind='simbad')

    seq = pancake.sequence.Sequence()
    seq.add_observation(target, exposures=obs['exposures'], nircam_mask='MASK210R', rolls=[0])
    seq.add_observation(reference, exposures=obs['exposures'], scale_exposures=target,
                        nircam_mask='MASK210R', nircam_sgd='5-POINT-BOX', min_ints=4)
    seqs.append(seq)

Target // Adding Source: Beta Pic


ValueError: Query to Simbad failed for Beta Pic, could not obtain: norm_val, norm_bandpass, norm_unit

In [ ]:
for seq in seqs:
    for obs in seq.observation_sequence:
        det = obs['configuration']['detector']
        print(f"{obs['scene'][0]['pancake_parameters']['name']:12s} "
              f"({obs['scene_name']}) "
              f"{obs['configuration']['instrument']['filter'].upper()}: "
              f"{det['readout_pattern'].upper()}, ngroup={det['ngroup']}, nint={det['nint']}")